In [67]:
import pandas as pd
# Load the datasets
df_suspects = pd.read_csv('suspects.csv')
#you need to type the file format (csv)
df_transactions = pd.read_csv('transactions.csv')

print("--- Suspects Head ---")
print(df_suspects.head())
#shows the first 5 rows

print("\n--- Transactions Head ---")
print(df_transactions.head())

print("--- Suspects Info ---")
df_suspects.info()
#,info() can tell you the missing data and data type and how many columns we have

print("\n--- Transactions Info ---")
df_transactions.info()

print("--- Transactions Bounds ---")
print(df_transactions.describe(include='all'))
#if you want to know all the information about this file you should conduct .describe(inclube='all')

--- Suspects Head ---
   suspect_id                  name                     occupation  \
0           1      William Jennings  Designer, television/film set   
1           2  Lesley Wilson-Newman       Engineer, communications   
2           3         Abdul Hopkins   Trade union research officer   
3           4      Mrs Lauren Green    Corporate investment banker   
4           5           Tina Foster         Fitness centre manager   

   nationality   age height_cm  weight_kg  
0      Germany  56.0       165       52.0  
1        Italy  69.0       188       90.0  
2       Canada  46.0       154       94.0  
3  Isle of Man  32.0      1.71       67.0  
4     Portugal  60.0       178       96.0  

--- Transactions Head ---
   transaction_id  suspect_id              date        amount        category  \
0            8640         188  15/10/2024 16:10  1.000000e+35          Retail   
1            3472          52  05/11/2024 23:42  4.970881e+04  Large Purchase   
2           19226      

In [68]:
df_transactions['date'] = pd.to_datetime(df_transactions['date'], dayfirst=True)
#pd.to_datetime makes 'data' from str to time, which python can reckon as a date number
#['date'] is for do the change to the specific columns
#dayfirst=true means python will see the first number as a day not month and only dayfirst can be used.
#format='%d-%m-%Y' equally dayfirst=True and Y have to be capital y

print(df_transactions.info())


<class 'pandas.DataFrame'>
RangeIndex: 18508 entries, 0 to 18507
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   transaction_id  18508 non-null  int64         
 1   suspect_id      18508 non-null  int64         
 2   date            18508 non-null  datetime64[us]
 3   amount          18508 non-null  float64       
 4   category        18501 non-null  str           
 5   recipient       18508 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(2), str(2)
memory usage: 867.7 KB
None


In [69]:
print("Missing values per column:")
print(df_transactions.isnull().sum())
#.isnull().sum() means count how many blanks we have and sum them as total
missing_category = df_transactions[df_transactions['category'].isnull()]
#go into the transactions table and then select the column in the transaction table and the isnull ones
print("\nTransactions with missing categories:")
print(missing_category)

Q1 = df_transactions['amount'].quantile(0.25)
Q3 = df_transactions['amount'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_transactions[(df_transactions['amount'] < lower_bound) | (df_transactions['amount'] > upper_bound)]

print(f"\nStatistical Outlier Limits: {lower_bound:.2f} to {upper_bound:.2f}")
print(f"Number of outlier transactions found: {len(outliers)}")

Missing values per column:
transaction_id    0
suspect_id        0
date              0
amount            0
category          7
recipient         0
dtype: int64

Transactions with missing categories:
       transaction_id  suspect_id                date  amount category  \
2323            14740         353 2024-11-11 09:33:00  220.53      NaN   
4363             2208          20 2024-10-04 08:23:00  193.40      NaN   
5029            19432         486 2024-09-28 06:07:00  185.02      NaN   
8312             2104          18 2024-10-22 01:44:00  140.54      NaN   
11156            2357          24 2024-11-12 09:43:00  103.20      NaN   
16402           14455         346 2024-09-23 07:51:00   32.82      NaN   
17783            2601          30 1924-11-03 11:47:00   14.30      NaN   

                          recipient  
2323        Cooper, Parker and Hall  
4363       Evans, Marsh and Baldwin  
5029                  Davey-Stevens  
8312   Tucker, Brown and O'Sullivan  
11156             

In [70]:
suspect_financials = df_transactions.groupby('suspect_id').agg({
    'amount': 'sum',
    'transaction_id': 'count'
})
#.groupby() makes a new table by 'suspect_id'
#.agg makes new columns and apply specific math
#the 'sum' will be shown in 'amount' column
suspect_financials = suspect_financials.rename(columns={
    'amount': 'total_spent',
    'transaction_id': 'num_transactions'
})
#.rename() can change the name of the columns
print(suspect_financials.head())

            total_spent  num_transactions
suspect_id                               
1               6494.72                53
2               4914.24                43
3               2887.32                24
4              32497.94                28
5               4355.72                30


In [71]:
df_master = pd.merge(df_suspects, suspect_financials, on='suspect_id', how='left')
#pd.merge will put suspect_financials into df_suspects
# on='suspect_id' makes python put the data from suspect_financials into the same name in df_suspects
# how is a keyword argument in python so the left table will be the basic table
df_master['total_spent'] = df_master['total_spent'].fillna(0)
# na means the cells shows nan
df_master['num_transactions'] = df_master['num_transactions'].fillna(0)

print(df_master.head())

   suspect_id                  name                     occupation  \
0           1      William Jennings  Designer, television/film set   
1           2  Lesley Wilson-Newman       Engineer, communications   
2           3         Abdul Hopkins   Trade union research officer   
3           4      Mrs Lauren Green    Corporate investment banker   
4           5           Tina Foster         Fitness centre manager   

   nationality   age height_cm  weight_kg  total_spent  num_transactions  
0      Germany  56.0       165       52.0      6494.72                53  
1        Italy  69.0       188       90.0      4914.24                43  
2       Canada  46.0       154       94.0      2887.32                24  
3  Isle of Man  32.0      1.71       67.0     32497.94                28  
4     Portugal  60.0       178       96.0      4355.72                30  


In [72]:
df_shortlist = df_master[df_master['num_transactions'] > 20]
# [df_master['num_transactions']>20] is a filter
remaining_count = len(df_shortlist)

print(f"Number of suspects remaining: {remaining_count}")
print("\n--- Final Suspect List ---")
for name in df_shortlist['name']:
    print(name)
# for name in df_shortlist['name'] makes python list the 'name' column
df_master.to_csv('master_suspects.csv',index=False)
# .to_csv() can save the data to the file
# index is the row numbers, =false can delete this column which we dont need it on our excel
print("Master DataFrame successfully saved as 'master_suspects.csv'.")

Number of suspects remaining: 431

--- Final Suspect List ---
William Jennings
Lesley Wilson-Newman
Abdul Hopkins
Mrs Lauren Green
Tina Foster
Nigel Edwards
Abdul Burton-Patterson
Sarah Reed-Walsh
Dr Lee Green
Eleanor Baker
Wayne Gough
Bernard Wilkinson-Simpson
Mrs Julia Harris
Dylan Young
Geoffrey Griffin
Megan Taylor
Miss Carole Woods
Ms Sheila Evans
Aaron Harrison
Brandon Hammond
Graham Wilkinson
Amanda Hall
Frances Holmes-Lloyd
Mrs Pauline Fuller
Dr Marc Phillips
Ruth Hawkins
Ms Jill Morley
Bernard Wilkins
Catherine Morris
Dr Paula Lloyd
Kathryn Hughes
Mr William Gould
Bethan Oliver
Lorraine Palmer
Arthur Smith
Christopher Davies
Bethan Young
Dr Danny Cook
Marion Steele-Harper
Alexander Taylor
Frances Buckley-Ryan
Ms Kayleigh Watson
David Anderson
John Griffiths
Miss Alice Tomlinson
Simon Cunningham
Joshua Stewart
Philip Clarke
Henry O'Brien
Leon Davison
Kayleigh Mitchell
Victor Patel
Eric O'Neill
Martyn Wilson
Sarah Hammond
Robert Wilson
Maria Bruce
Olivia King
Kayleigh Daniels
Ge

In [73]:
df_suspects['height_cm'] = pd.to_numeric(df_suspects['height_cm'], errors='coerce')
df_suspects['weight_kg'] = pd.to_numeric(df_suspects['weight_kg'], errors='coerce')

df_suspects['bmi'] = df_suspects['weight_kg'] / (df_suspects['height_cm'] / 100) ** 2

print(df_suspects[['height_cm', 'weight_kg', 'bmi']].head())

   height_cm  weight_kg            bmi
0     165.00       52.0      19.100092
1     188.00       90.0      25.464011
2     154.00       94.0      39.635689
3       1.71       67.0  229130.330700
4     178.00       96.0      30.299205


In [74]:
df_suspects['name'] = df_suspects['name'].str.strip()
name_split = df_suspects['name'].str.split(' ',n=1,expand=True)
df_suspects['first_name'] = name_split[0]
df_suspects['surname'] = name_split[1]
print(df_suspects[['name', 'first_name', 'surname']].head())

                   name first_name        surname
0      William Jennings    William       Jennings
1  Lesley Wilson-Newman     Lesley  Wilson-Newman
2         Abdul Hopkins      Abdul        Hopkins
3      Mrs Lauren Green        Mrs   Lauren Green
4           Tina Foster       Tina         Foster


In [75]:
df_transactions['day_name'] = df_transactions['date'].dt.day_name()
print(df_transactions['day_name'].head())

0      Tuesday
1      Tuesday
2    Wednesday
3    Wednesday
4       Monday
Name: day_name, dtype: str
